Notebook purpose

## Initialization

In [ ]:
# Importing needed code

import typing
import time

from data_processing import helpers
from data_processing import loading

### Functions

In [ ]:
T = typing.TypeVar("T")
P = typing.ParamSpec("P")
Fn = typing.Callable[P, T]


def time_function(
    fn: Fn, *fn_args: P.args, **fn_kwargs: P.kwargs
) -> tuple[T, float]:
    # record start_time
    start_time = time.time()
    results = fn(*fn_args, **fn_kwargs)
    # record end_time and print
    end_time = time.time()
    elapsed = end_time - start_time
    return results, elapsed

In [ ]:
def time_loading_csv(benchmark_time_dict):
    for exp_id, exp_times in benchmark_time_dict.items():
        df, elapsed = time_function(loading.load_caen_csvs, exp_id)
        exp_times["csv"] = (df.shape[0], elapsed)
        return benchmark_time_dict

In [ ]:
def time_loading_parquets(benchmark_time_dict):
    for exp_id, exp_times in benchmark_time_dict.items():
        df, elapsed = time_function(loading.load_parquet_psd, exp_id)
        exp_times["parquet"] = (df.shape[0], elapsed)
        return benchmark_time_dict

In [ ]:
def report_time_loading(benchmark_time_dict):
    for exp_id, benchmark_data in benchmark_time_dict.items():
        parquet_data = benchmark_data["parquet"]
        csv_data = benchmark_data["csv"]
        pq_rows, pq_elapsed = parquet_data
        csv_rows, csv_elapsed = csv_data
        print(f"---{exp_id}---")
        print(f"Parquet: {pq_elapsed:6.2f} s for {pq_rows:n} rows")
        print(f"CSV:     {csv_elapsed:6.2f} s for {csv_rows:n} rows")

## User Inputs

In [ ]:
experiment_ids = helpers.input_experiment_ids()

## Analysis

In [ ]:
for _ in range(3):
    benchmark_times = {exp_id: {} for exp_id in experiment_ids}
    benchmark_times = time_loading_csv(benchmark_times)
    benchmark_times = time_loading_parquets(benchmark_times)
    report_time_loading(benchmark_times)